# Notebook 05: Text Overlay + Font Rendering

**Goal:** Render translated text onto inpainted images at the correct positions, with appropriate Indic script fonts, auto-sized to fit the original bounding boxes.

**Steps:**
1. Download and load Noto Sans fonts for all 5 Indic scripts
2. Fit translated text into bounding boxes (auto-shrink + word wrap)
3. Detect text and background colors
4. Render translated text onto inpainted images

**Input:** Inpainted images from `data/inpainted/best/` + translations from `data/translations/best/`  
**Output:** Final translated images saved in `data/output/`

In [ ]:
# Install dependencies (run once)
# !pip install Pillow numpy matplotlib
# !python scripts/download_fonts.py

In [ ]:
import json
import os
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    DATA_DIR, PAGE_IMAGES_DIR, INPAINTED_DIR, TRANSLATIONS_DIR,
    OUTPUT_DIR, FONTS_DIR,
    load_json, save_json, load_image, save_image,
    display_images, display_comparison,
    sample_background_color, estimate_text_color,
)

# Load inputs
PAGE_INDEX = 0
TARGET_LANGUAGE = "hindi"  # Must match what was used in Notebook 03

inpainted_image = load_image(INPAINTED_DIR / "best" / f"page_{PAGE_INDEX}.png")
original_image = load_image(PAGE_IMAGES_DIR / f"page_{PAGE_INDEX}.png")
translations = load_json(TRANSLATIONS_DIR / "best" / f"page_{PAGE_INDEX}_{TARGET_LANGUAGE}.json")

print(f"Inpainted image: {inpainted_image.size[0]}x{inpainted_image.size[1]}")
print(f"Translations: {len(translations)} blocks")
print(f"Target language: {TARGET_LANGUAGE}")

## Font Manager

Map each target language to its corresponding Noto Sans font. Download fonts if not present.

In [ ]:
# Language → font file mapping
LANGUAGE_FONT_MAP = {
    "hindi":    "NotoSansDevanagari-Regular.ttf",
    "tamil":    "NotoSansTamil-Regular.ttf",
    "bengali":  "NotoSansBengali-Regular.ttf",
    "punjabi":  "NotoSansGurmukhi-Regular.ttf",
    "gujarati": "NotoSansGujarati-Regular.ttf",
}


def get_font(language: str, size: int = 24) -> ImageFont.FreeTypeFont:
    """Load the appropriate Noto Sans font for the target language."""
    font_file = LANGUAGE_FONT_MAP.get(language)
    if not font_file:
        raise ValueError(f"No font configured for language: {language}")
    
    font_path = FONTS_DIR / font_file
    if not font_path.exists():
        # Try to download fonts
        print(f"Font not found: {font_path}. Running download script...")
        import subprocess
        subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "download_fonts.py")], check=True)
    
    if not font_path.exists():
        print(f"WARNING: Font not found at {font_path}, falling back to default")
        return ImageFont.load_default()
    
    return ImageFont.truetype(str(font_path), size=size)


# Test font loading
test_font = get_font(TARGET_LANGUAGE, size=30)
print(f"Font loaded: {LANGUAGE_FONT_MAP[TARGET_LANGUAGE]} at size 30")

## Text Fitter

Auto-sizes text to fit within the bounding box. Reduces font size iteratively and wraps text to fit.

In [ ]:
def wrap_text_to_bbox(text: str, font: ImageFont.FreeTypeFont, max_width: int) -> list[str]:
    """
    Word-wrap text to fit within max_width pixels.
    
    Handles Indic scripts where spaces may be less common by also
    breaking on very long words that exceed the width.
    """
    words = text.split()
    if not words:
        return []
    
    tmp_draw = ImageDraw.Draw(Image.new("RGB", (1, 1)))
    lines = []
    current_line = ""
    
    for word in words:
        if not current_line:
            test_line = word
        else:
            test_line = current_line + " " + word
        
        line_width = tmp_draw.textbbox((0, 0), test_line, font=font)[2]
        
        if line_width <= max_width:
            current_line = test_line
        else:
            if current_line:
                lines.append(current_line)
            # If a single word is wider than the box, force it on its own line
            current_line = word
    
    if current_line:
        lines.append(current_line)
    
    return lines


def fit_text_in_bbox(
    text: str,
    bbox: list[int],
    language: str,
    min_font_size: int = 6,
) -> tuple[ImageFont.FreeTypeFont, list[str], int]:
    """
    Find the best font size and line wrapping to fit text in a bounding box.
    
    Translated text (especially Hindi) can be 30-50% longer than English,
    so we shrink aggressively from the estimated original size down to min_font_size.
    
    Args:
        text: The translated text to render
        bbox: [x0, y0, x1, y1] bounding box
        language: Target language for font selection
        min_font_size: Absolute minimum font size (pixels)
    
    Returns:
        (font, wrapped_lines, font_size)
    """
    x0, y0, x1, y1 = bbox
    box_width = x1 - x0
    box_height = y1 - y0
    
    if box_width <= 0 or box_height <= 0:
        font = get_font(language, size=min_font_size)
        return font, [text], min_font_size
    
    # Start from estimated original font size
    initial_size = max(min_font_size, int(box_height * 0.75))
    
    tmp_draw = ImageDraw.Draw(Image.new("RGB", (1, 1)))
    
    # Try decreasing font sizes until text fits
    for font_size in range(initial_size, min_font_size - 1, -1):
        font = get_font(language, size=font_size)
        lines = wrap_text_to_bbox(text, font, box_width)
        
        if not lines:
            continue
        
        # Calculate total text height
        line_height = tmp_draw.textbbox((0, 0), text[:5] or "A", font=font)[3]
        line_spacing = max(1, int(font_size * 0.15))
        total_height = line_height * len(lines) + line_spacing * (len(lines) - 1)
        
        if total_height <= box_height:
            return font, lines, font_size
    
    # At minimum size — truncate lines that don't fit vertically
    font = get_font(language, size=min_font_size)
    lines = wrap_text_to_bbox(text, font, box_width)
    line_height = tmp_draw.textbbox((0, 0), text[:5] or "A", font=font)[3]
    line_spacing = max(1, int(min_font_size * 0.15))
    max_lines = max(1, box_height // (line_height + line_spacing))
    lines = lines[:max_lines]
    
    return font, lines, min_font_size


# Test the fitter on a sample block
if translations:
    sample = translations[0]
    font, lines, size = fit_text_in_bbox(
        sample["translated"], sample["bbox"], TARGET_LANGUAGE
    )
    print(f"Sample text: {sample['translated'][:50]}...")
    print(f"Bbox: {sample['bbox']} → w={sample['bbox'][2]-sample['bbox'][0]}, h={sample['bbox'][3]-sample['bbox'][1]}")
    print(f"Fitted to font size: {size}")
    print(f"Lines: {len(lines)}")
    for l in lines:
        print(f"  '{l}'")

## Render Translated Text

Draw all translated text blocks onto the inpainted image, producing the final output.

In [ ]:
def render_translated_text(
    image: Image.Image,
    original_image: Image.Image,
    translations: list[dict],
    language: str,
) -> Image.Image:
    """
    Render translated text onto the inpainted image.
    
    Uses clipping to ensure text never overflows its bounding box.
    """
    result = image.copy()
    tmp_draw = ImageDraw.Draw(Image.new("RGB", (1, 1)))
    
    for trans in translations:
        text = trans["translated"]
        bbox = trans["bbox"]
        x0, y0, x1, y1 = [int(c) for c in bbox]
        box_w, box_h = x1 - x0, y1 - y0
        
        if not text.strip() or box_w <= 0 or box_h <= 0:
            continue
        
        # Detect text color from original image
        bg_color = sample_background_color(original_image, [x0, y0, x1, y1])
        text_color = estimate_text_color(original_image, [x0, y0, x1, y1], bg_color)
        
        # Fit text to bbox
        font, lines, font_size = fit_text_in_bbox(text, [x0, y0, x1, y1], language)
        
        if not lines:
            continue
        
        # Draw text onto a separate image and paste with clipping
        # This prevents any overflow beyond the bounding box
        text_img = Image.new("RGBA", (box_w, box_h), (0, 0, 0, 0))
        text_draw = ImageDraw.Draw(text_img)
        
        line_height = tmp_draw.textbbox((0, 0), lines[0], font=font)[3]
        line_spacing = max(1, int(font_size * 0.15))
        total_height = line_height * len(lines) + line_spacing * (len(lines) - 1)
        
        # Vertical centering within the box
        y_offset = max(0, (box_h - total_height) // 2)
        
        for line in lines:
            if y_offset + line_height > box_h:
                break  # Stop if we'd overflow vertically
            text_draw.text((0, y_offset), line, fill=text_color, font=font)
            y_offset += line_height + line_spacing
        
        # Paste the clipped text onto the result
        result.paste(text_img, (x0, y0), mask=text_img)
    
    return result


# Render translated text
output_image = render_translated_text(inpainted_image, original_image, translations, TARGET_LANGUAGE)

print("Rendering complete!")
display_comparison(original_image, output_image, "Original (English)", f"Translated ({TARGET_LANGUAGE.title()})")

## Save Output

In [ ]:
# Save the final output
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / f"page_{PAGE_INDEX}_{TARGET_LANGUAGE}.png"
save_image(output_image, output_path)

print(f"Saved: {output_path}")
print(f"\n✓ Text overlay complete. Next notebook: 06_full_pipeline.ipynb")